In [39]:

from pyspark.sql import SparkSession
spark = (
    SparkSession.builder
    .appName("GTFS_Timetable_Preprocessing")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# reducing unnecessary Spark log messages.
spark.sparkContext.setLogLevel("WARN")
print("PySpark version:", spark.version)
print(
    "Shuffle partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)
print("Spark UI:", spark.sparkContext.uiWebUrl)

PySpark version: 3.5.1
Shuffle partitions: 4
Spark UI: http://192.168.1.7:4041


In [40]:

from pathlib import Path
#  folder where the notebook is currently running.
current_folder = Path.cwd()
if current_folder.name == "notebooks":
    project_root = current_folder.parent
else:
    project_root = current_folder
gtfs_folder = (
    project_root
    / "data"
    / "raw"
    / "timetable"
    / "extracted"
)

#  where the processed timetable will be saved.
processed_folder = (
    project_root
    / "data"
    / "processed"
    / "timetable"
)

print("Current folder:", current_folder)
print("Project root:", project_root)
print("GTFS folder:", gtfs_folder)
print("GTFS folder exists:", gtfs_folder.exists())

Current folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/notebooks
Project root: /Users/babitaadhikari/Desktop/bus-disruption-platform
GTFS folder: /Users/babitaadhikari/Desktop/bus-disruption-platform/data/raw/timetable/extracted
GTFS folder exists: True


In [41]:
# main GTFS files required for timetable analysis.
required_files = [
    "agency.txt",
    "routes.txt",
    "trips.txt",
    "stop_times.txt",
    "stops.txt"
]
missing_files = []
for filename in required_files:
    file_path = gtfs_folder / filename

    if file_path.exists():
        print("Found:", filename)
    else:
        missing_files.append(filename)

# stoop notebook when an important file is missing.
if missing_files:
    raise FileNotFoundError(
        f"Missing GTFS files: {missing_files}"
    )

print("All required GTFS files are available.")

Found: agency.txt
Found: routes.txt
Found: trips.txt
Found: stop_times.txt
Found: stops.txt
All required GTFS files are available.


## GTFS loading function

In [42]:
def load_gtfs_file(filename):
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("quote", '"')
        .option("escape", '"')
        .option("mode", "PERMISSIVE")
        .csv(str(gtfs_folder / filename))
    )
print("GTFS loader created successfully.")

GTFS loader created successfully.


## Load timetable tables

In [43]:
#  operator information
agency_df = load_gtfs_file("agency.txt")
#  route information
routes_df = load_gtfs_file("routes.txt")
#  scheduled journey information.
trips_df = load_gtfs_file("trips.txt")
#  arrival and departure time for every trip and stop
stop_times_df = load_gtfs_file("stop_times.txt")
#  stop names and geographical coordinates
stops_df = load_gtfs_file("stops.txt")
print("All GTFS tables loaded successfully.")

All GTFS tables loaded successfully.


In [44]:
# storing all DataFrames in a dictionary so they can be checked together
datasets = {
    "agency": agency_df,
    "routes": routes_df,
    "trips": trips_df,
    "stop_times": stop_times_df,
    "stops": stops_df
}
dataset_counts = {}
#  rows and inspect the initial number of partitions
for name, dataframe in datasets.items():
    row_count = dataframe.count()
    partition_count = dataframe.rdd.getNumPartitions()
    dataset_counts[name] = row_count
    print(
        f"{name}: {row_count:,} rows "
        f"| {partition_count} partitions"
    )

agency: 60 rows | 1 partitions
routes: 1,171 rows | 1 partitions
trips: 135,003 rows | 5 partitions


stop_times: 5,370,199 rows | 8 partitions
stops: 0 rows | 1 partitions


In [45]:
# showing the column names in every GTFS table.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()} COLUMNS")
    print(dataframe.columns)


AGENCY COLUMNS
['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'agency_noc']

ROUTES COLUMNS
['route_id', 'agency_id', 'route_short_name', 'route_long_name', 'route_type']

TRIPS COLUMNS
['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'vehicle_journey_code']

STOP_TIMES COLUMNS
['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'shape_dist_traveled', 'timepoint']

STOPS COLUMNS
['stop_id', 'stop_code', 'stop_name', 'stop_lat', 'stop_lon', 'wheelchair_boarding', 'location_type', 'parent_station', 'platform_code']


In [ ]:
print("AGENCY SAMPLE")
agency_df.show(5, truncate=False)
print("ROUTES SAMPLE")
routes_df.show(5, truncate=False)
print("TRIPS SAMPLE")
trips_df.show(5, truncate=False)
print("STOP TIMES SAMPLE")
stop_times_df.show(5, truncate=False)
print("STOPS SAMPLE")
stops_df.show(5, truncate=False)

AGENCY SAMPLE


+---------+-------------------------+--------------------------+---------------+-----------+------------+----------+
|agency_id|agency_name              |agency_url                |agency_timezone|agency_lang|agency_phone|agency_noc|
+---------+-------------------------+--------------------------+---------------+-----------+------------+----------+
|OP101    |Select Bus Services      |https://www.traveline.info|Europe/London  |EN         |NULL        |SLBS      |
|OP102    |LandFlight               |https://www.traveline.info|Europe/London  |EN         |NULL        |SLVL      |
|OP10232  |Solus Coaches            |https://www.traveline.info|Europe/London  |EN         |NULL        |SOLU      |
|OP103    |National Express Coventry|https://www.traveline.info|Europe/London  |EN         |NULL        |TCVW      |
|OP104    |Let's Go                 |https://www.traveline.info|Europe/London  |EN         |NULL        |TEXP      |
+---------+-------------------------+--------------------------+

+--------+---------+----------------+---------------+----------+
|route_id|agency_id|route_short_name|route_long_name|route_type|
+--------+---------+----------------+---------------+----------+
|211     |OP850    |740             |NULL           |3         |
|541     |OP87     |X15             |NULL           |3         |
|542     |OP87     |X75             |NULL           |3         |
|555     |OP90     |226             |NULL           |3         |
|560     |OP90     |26A             |NULL           |3         |
+--------+---------+----------------+---------------+----------+
only showing top 5 rows

TRIPS SAMPLE


+--------+----------+------------------------------------------+--------------+------------+--------+--------+---------------------+--------------------+
|route_id|service_id|trip_id                                   |trip_headsign |direction_id|block_id|shape_id|wheelchair_accessible|vehicle_journey_code|
+--------+----------+------------------------------------------+--------------+------------+--------+--------+---------------------+--------------------+
|211     |97        |VJ448a5adb0d827eaa35def099a63134cb15aa7011|Assembly Rooms|0           |NULL    |NULL    |0                    |vj_3                |
|211     |97        |VJbc68342ea11436266488037018d8fac7d73cf381|Bus Station   |1           |NULL    |NULL    |0                    |vj_4                |
|211     |97        |VJbce3b63a0c4e4c183e0bfe4f9252dc8c29a4c7f8|Assembly Rooms|0           |NULL    |NULL    |0                    |vj_2                |
|211     |97        |VJddae15c0f3825bc75eed82a58d1791cb99fbf4a1|Assembly Roo

[Stage 330:>                                                        (0 + 1) / 1]

In [ ]:
# schema of every loaded DataFrame.
for name, dataframe in datasets.items():
    print(f"\n{name.upper()} SCHEMA")
    dataframe.printSchema()

## checking missing values

In [ ]:
from pyspark.sql.functions import (
    col,
    trim,
    when,
    sum as spark_sum
)
def show_missing_values(dataframe, columns, dataset_name):
    """
    Count null and blank values in selected columns.
    """
    #  examine columns that actually exist
    available_columns = [
        column_name
        for column_name in columns
        if column_name in dataframe.columns
    ]
    expressions = []
    for column_name in available_columns:
        missing_condition = (
            col(column_name).isNull()
            | (trim(col(column_name)) == "")
        )
        expressions.append(
            spark_sum(
                when(missing_condition, 1).otherwise(0)
            ).alias(column_name)
        )
    print(f"Missing values in {dataset_name}:")
    dataframe.select(expressions).show(
        truncate=False
    )
show_missing_values(
    agency_df,
    ["agency_id", "agency_name", "agency_noc"],
    "agency"
)
show_missing_values(
    routes_df,
    [
        "route_id",
        "agency_id",
        "route_short_name",
        "route_long_name"
    ],
    "routes"
)
show_missing_values(
    trips_df,
    [
        "trip_id",
        "route_id",
        "service_id",
        "vehicle_journey_code"
    ],
    "trips"
)
show_missing_values(
    stop_times_df,
    [
        "trip_id",
        "stop_id",
        "arrival_time",
        "departure_time",
        "stop_sequence"
    ],
    "stop_times"
)
show_missing_values(
    stops_df,
    [
        "stop_id",
        "stop_name",
        "stop_lat",
        "stop_lon"
    ],
    "stops"
)

## checking duplicates

In [ ]:
def duplicate_key_count(dataframe, key_columns):
    total_count = dataframe.count()
    unique_count = (
        dataframe
        .select(*key_columns)
        .dropDuplicates()
        .count()
    )
    return total_count - unique_count
print(
    "Duplicate agency IDs:",
    duplicate_key_count(
        agency_df,
        ["agency_id"]
    )
)
print(
    "Duplicate route IDs:",
    duplicate_key_count(
        routes_df,
        ["route_id"]
    )
)
print(
    "Duplicate trip IDs:",
    duplicate_key_count(
        trips_df,
        ["trip_id"]
    )
)
print(
    "Duplicate stop IDs:",
    duplicate_key_count(
        stops_df,
        ["stop_id"]
    )
)
print(
    "Duplicate trip-stop sequences:",
    duplicate_key_count(
        stop_times_df,
        ["trip_id", "stop_sequence"]
    )
)

In [ ]:
def trim_string_columns(dataframe):
    cleaned_df = dataframe
    for column_name, data_type in dataframe.dtypes:
        if data_type == "string":
            cleaned_df = cleaned_df.withColumn(
                column_name,
                trim(col(column_name))
            )
    return cleaned_df
# clean leading and trailing spaces
agency_clean = trim_string_columns(agency_df)
routes_clean = trim_string_columns(routes_df)
trips_clean = trim_string_columns(trips_df)
stop_times_clean = trim_string_columns(stop_times_df)
stops_clean = trim_string_columns(stops_df)
print("String columns cleaned successfully.")

In [ ]:
# keep one record for each operator
agency_clean = agency_clean.dropDuplicates(
    ["agency_id"]
)
# keep one record for each route
routes_clean = routes_clean.dropDuplicates(
    ["route_id"]
)
# keep one record for each trip
trips_clean = trips_clean.dropDuplicates(
    ["trip_id"]
)
# keep one record for each stop
stops_clean = stops_clean.dropDuplicates(
    ["stop_id"]
)
#  trip can visit many stops, so the combined key is used
stop_times_clean = stop_times_clean.dropDuplicates(
    ["trip_id", "stop_sequence"]
)
print("Duplicate key records removed.")

## convert numeric columns

In [ ]:
# convert route type when the column exists
if "route_type" in routes_clean.columns:
    routes_clean = routes_clean.withColumn(
        "route_type",
        col("route_type").cast("int")
    )
# change stop sequence into an integer
stop_times_clean = stop_times_clean.withColumn(
    "stop_sequence",
    col("stop_sequence").cast("int")
)
# convert stop coordinates into numeric values
stops_clean = (
    stops_clean
    .withColumn(
        "stop_lat",
        col("stop_lat").cast("double")
    )
    .withColumn(
        "stop_lon",
        col("stop_lon").cast("double")
    )
)
print("Important numeric columns converted.")

## GTFS times into seconds

In [ ]:
from pyspark.sql.functions import split
# splitting arrival time into hours, minutes and seconds
arrival_parts = split(
    col("arrival_time"),
    ":"
)
#  departure time into hours, minutes and seconds
departure_parts = split(
    col("departure_time"),
    ":"
)
stop_times_clean = (
    stop_times_clean
    .withColumn(
        "arrival_seconds",
        arrival_parts.getItem(0).cast("int") * 3600
        + arrival_parts.getItem(1).cast("int") * 60
        + arrival_parts.getItem(2).cast("int")
    )
    .withColumn(
        "departure_seconds",
        departure_parts.getItem(0).cast("int") * 3600
        + departure_parts.getItem(1).cast("int") * 60
        + departure_parts.getItem(2).cast("int")
    )
)
stop_times_clean.select(
    "trip_id",
    "stop_id",
    "arrival_time",
    "arrival_seconds",
    "departure_time",
    "departure_seconds"
).show(5, truncate=False)

## Join trips and routes

In [ ]:
from pyspark.sql.functions import broadcast
# routes is much smaller than trips, so Spark can broadcast it
trip_route_df = (
    trips_clean
    .join(
        broadcast(routes_clean),
        on="route_id",
        how="inner"
    )
)
print(
    "Trips joined with routes:",
    f"{trip_route_df.count():,}"
)

## Join operator information

In [ ]:
# adding operator information when agency_id exists
if (
    "agency_id" in trip_route_df.columns
    and "agency_id" in agency_clean.columns
):
    trip_details_df = (
        trip_route_df
        .join(
            broadcast(agency_clean),
            on="agency_id",
            how="left"
        )
    )
else:
    # continuing without agency join when agency_id is unavailable
    trip_details_df = trip_route_df

print(
    "Trip detail rows:",
    f"{trip_details_df.count():,}"
)

## Join stop times

In [ ]:
# join every scheduled stop time with its trip and route details
timetable_df = (
    stop_times_clean
    .join(
        trip_details_df,
        on="trip_id",
        how="inner"
    )
)
print("Stop times joined with trip details.")

## Join stop names and coordinates

In [ ]:
# stops is small enough to be broadcast to the larger timetable table
timetable_df = (
    timetable_df
    .join(
        broadcast(stops_clean),
        on="stop_id",
        how="left"
    )
)
print("Stops joined successfully.")

In [ ]:
#  most useful columns for later integration
# with live vehicle-location data
preferred_columns = [
    "agency_id",
    "agency_name",
    "agency_noc",
    "route_id",
    "route_short_name",
    "route_long_name",
    "service_id",
    "trip_id",
    "vehicle_journey_code",
    "trip_headsign",
    "direction_id",
    "stop_id",
    "stop_code",
    "stop_name",
    "stop_sequence",
    "arrival_time",
    "departure_time",
    "arrival_seconds",
    "departure_seconds",
    "stop_lat",
    "stop_lon",
    "shape_id"
]
#  only columns that are actually available in this dataset
available_columns = [
    column_name
    for column_name in preferred_columns
    if column_name in timetable_df.columns
]
timetable_df = timetable_df.select(
    *available_columns
)
print("Selected timetable columns:")
print(timetable_df.columns)

In [ ]:
# redistribute the joined records across four Spark partitions
timetable_df = timetable_df.repartition(
    4,
    "route_id"
)
print(
    "Partitions after repartition:",
    timetable_df.rdd.getNumPartitions()
)

## Cache and count

In [ ]:
# cache the joined timetable because it will be reused
timetable_df.cache()
# count() triggers Spark's lazy execution and places data in cache
joined_count = timetable_df.count()
print(
    "Joined timetable rows:",
    f"{joined_count:,}"
)
print(
    "Cached partitions:",
    timetable_df.rdd.getNumPartitions()
)

In [ ]:
timetable_df.show(
    10,
    truncate=False
)

In [ ]:
print(
    "Distinct routes:",
    timetable_df.select(
        "route_id"
    ).distinct().count()
)
print(
    "Distinct trips:",
    timetable_df.select(
        "trip_id"
    ).distinct().count()
)
print(
    "Distinct stops:",
    timetable_df.select(
        "stop_id"
    ).distinct().count()
)
if "agency_id" in timetable_df.columns:
    print(
        "Distinct operators:",
        timetable_df.select(
            "agency_id"
        ).distinct().count()
    )

## PySpark SQL query for joining

In [ ]:
# registering the DataFrame as a temporary SQL table
timetable_df.createOrReplaceTempView(
    "timetable"
)
# SQL Query 1:
# count scheduled stop records and scheduled trips for each route
route_schedule_summary = spark.sql("""
    SELECT
        route_id,
        route_short_name,
        COUNT(*) AS scheduled_stop_records,
        COUNT(DISTINCT trip_id) AS scheduled_trips
    FROM timetable
    GROUP BY route_id, route_short_name
    ORDER BY scheduled_stop_records DESC
    LIMIT 10
""")
route_schedule_summary.show(
    truncate=False
)

In [ ]:
# showing how Spark plans to execute the timetable operations
timetable_df.explain(
    mode="formatted"
)

In [ ]:
#   processed timetable folder if it does not exist
processed_folder.mkdir(
    parents=True,
    exist_ok=True
)
# saving cleaned and joined timetable DataFrame
# overwrite mode replaces the previous output when the cell is rerun
(
    timetable_df.write
    .mode("overwrite")
    .parquet(str(processed_folder))
)
print("Processed timetable saved successfully.")
print("Saved location:", processed_folder)

In [ ]:
# read the saved Parquet files back into PySpark
saved_timetable_df = spark.read.parquet(
    str(processed_folder)
)
# count the saved records
saved_count = saved_timetable_df.count()
print(
    "Original joined rows:",
    f"{joined_count:,}"
)
print(
    "Saved Parquet rows:",
    f"{saved_count:,}"
)
print(
    "Saved Parquet partitions:",
    saved_timetable_df.rdd.getNumPartitions()
)
# check whether any records were lost while saving
if saved_count != joined_count:
    raise ValueError(
        "Saved row count does not match the original joined row count."
    )
print("Parquet verification successful.")

In [ ]:
# display the files created inside the processed timetable folder
parquet_files = sorted(
    processed_folder.glob("*")
)
print("Files created:", len(parquet_files))
for file_path in parquet_files:
    print(file_path.name)

In [ ]:
#  final summary of the processed timetable
print("TIMETABLE PREPROCESSING SUMMARY")
print("-" * 40)
print("Total rows:", f"{joined_count:,}")
print(
    "Distinct routes:",
    timetable_df.select("route_id").distinct().count()
)
print(
    "Distinct trips:",
    timetable_df.select("trip_id").distinct().count()
)
print(
    "Distinct stops:",
    timetable_df.select("stop_id").distinct().count()
)
print(
    "Partitions:",
    timetable_df.rdd.getNumPartitions()
)
print("Saved format: Parquet")
print("Saved location:", processed_folder)